In [ ]:
# @title # **Stage 0.** _Storage_

from IPython.display import clear_output
import os

def connect_to_drive(drive_path, cloud_dir):
  if os.path.exists(drive_path):
    return None

  try:
    from google.colab import drive

    drive.mount(drive_path, force_remount=False)
    os.makedirs(cloud_dir, exist_ok=True)
    clear_output()
  except:
    ...

  del drive

invokeai_dir = "/content/db"
drive_path = "/content/drive"
cloud_dir = f"{drive_path}/MyDrive/InvokeAI/db"

connect_to_drive(drive_path, cloud_dir)

In [ ]:
# @title # **Stage 1.** _Initializing Functions_

from IPython.display import clear_output
import os

include_dirs = [
    'configs',
    'databases',
    'keys',
    'models/any',
    'models/sd-1',
    'models/sd-2',
    'nodes'
] # Добавьте директории если нужно

include_compressed_dirs = [
    'outputs'
] # Добавьте директории если нужно

def load_from(src, dst): # Исправить в будущем
  import distutils.dir_util

  try:
    distutils.dir_util.copy_tree(
        src,
        dst,
        update=1,
        verbose=1
    )
  except Exception as e:
    print(f"[1;31mОшибка : {e}")

  for compress_dir in include_compressed_dirs:
    src_path = f"{dst}/{compress_dir}"

    if os.path.exists(f"{src_path}.zip"):
      !unzip {src_path} -d {dst} && rm {src_path}.zip

def save_to(src, dst): # Исправить в будущем

  from distutils import dir_util

  for dir_to_copy in include_dirs:
    try:
      src_path = f"{src}/{dir_to_copy}"
      dst_path = f"{dst}/{dir_to_copy}"

      dir_util.copy_tree(
          src_path,
          dst_path,
          update=1,
          verbose=1,
      )
    except Exception as e:
      print(f"[1;31mОшибка : {e}")

  import filecmp
  import shutil

  for dir_to_copy in include_compressed_dirs:
    src_path = f"{src}/{dir_to_copy}"
    dst_path = f"{dst}/{dir_to_copy}"

    shutil.make_archive(src_path, "zip", src, dir_to_copy)
    try:
      if not os.path.exists(f"{dst_path}.zip") or not filecmp.cmp(f"{src_path}.zip", f"{dst_path}.zip"):
        shutil.copyfile(f"{src_path}.zip", f"{dst_path}.zip")
        !rm {src_path}.zip
    except Exception as e:
      print(f"[1;31mОшибка : {e}")

def install_invokeai(ver):
  %cd /content

  dir_name = 'InvokeAI'

  if not os.path.exists(dir_name):
    !wget github.com/invoke-ai/{dir_name}/archive/refs/tags/{ver}.zip
    !unzip {ver}.zip && rm {ver}.zip
    !mv ./{dir_name}-{ver[1:]} ./{dir_name}

  clear_output()

def remove_sample_data():
  !rm -rf ./sample_data
  clear_output()

def install_python_venv():
  !apt install python3.10-venv
  clear_output()

def install_dependencies(invokeai_dir):
  %cd /content/InvokeAI/
  !./installer/install.sh.in -r {invokeai_dir} -y

  %cd /content/InvokeAI/
  !pip install -q -e .
  clear_output()

def build_webui():
  %cd /content/InvokeAI/invokeai/frontend/web
  !npm install -g n
  !n stable
  !npm install -g pnpm
  !pnpm install
  !pnpm vite build
  clear_output()

def install_models(invokeai_dir):
  %cd /content/InvokeAI/

  models_config_path = f"{invokeai_dir}/configs/INITIAL_MODELS.yaml"

  if os.path.exists(models_config_path):
    !cp {models_config_path} ./invokeai/configs/

  !python scripts/configure_invokeai.py --root_dir {invokeai_dir} -y

  clear_output()

In [ ]:
# @title # **Step 2.** _Installation_

install_invokeai('v3.6.0rc4')
install_python_venv()
load_from(cloud_dir, invokeai_dir)
install_dependencies(invokeai_dir)
build_webui()
remove_sample_data()

clear_output()

In [ ]:
# @title # **Stage 3**. _Downdload Models_

install_models(invokeai_dir)

In [ ]:
# @title # **Stage 4**. _Launch InvokeAI_

from IPython.display import clear_output
from pathlib import Path
from typing import Union
import subprocess
import os

keys_dir = f"{invokeai_dir}/keys"
id_rsa_file = f"{keys_dir}/id_rsa"
id_rsa_pub_file = f"{keys_dir}/id_rsa.pub"

def generate_key() -> None:
  id_rsa_file_exists = os.path.exists(id_rsa_file)
  id_rsa_pub_file_exists = os.path.exists(id_rsa_pub_file)

  if not id_rsa_file_exists or not id_rsa_pub_file_exists:
    if id_rsa_file_exists:
      os.remove(id_rsa_file)
    if id_rsa_pub_file_exists:
      os.remove(id_rsa_pub_file)

    if not os.path.exists(keys_dir):
      os.makedirs(keys_dir, exist_ok=True)

    path = Path(id_rsa_file)
    args = ['ssh-keygen', '-t', 'rsa', '-b', '4096', '-N', '', '-q', '-f', path.as_posix()]
    subprocess.run(args, check=True)
    path.chmod(0o600)

def create_tunnel():
  import threading

  def tunnel():
    !ssh -R 80:127.0.0.1:9090 -o StrictHostKeyChecking=no -i {id_rsa_file} remote.moe

  threading.Thread(target=tunnel, daemon=True).start()

def start_invokeai(invokeai_dir):
  %cd /content/InvokeAI
  !python scripts/invokeai-web.py --root {invokeai_dir}
  clear_output()

generate_key()
create_tunnel()
start_invokeai(invokeai_dir)
save_to(invokeai_dir, cloud_dir)